# Training Pipeline
Use this notebook after text and image embeddings are generated. Run the cells sequentially on the GPU-enabled environment.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
from sklearn.model_selection import KFold
import joblib

import os
os.chdir(r"d:\Projects\Amazon_ML_Challenge")
DATA_DIR = Path("dataset")
OUTPUT_DIR = Path("outputs")
FEATURE_DIR = Path("features")
MODEL_DIR = Path("models")
SUBMISSION_PATH = MODEL_DIR / "test_out.csv"

TEXT_TRAIN_PATH = OUTPUT_DIR / "train_text_embeddings.npy"
TEXT_TEST_PATH = OUTPUT_DIR / "test_text_embeddings.npy"
IMAGE_TRAIN_PATH = OUTPUT_DIR / "train_image_embeddings.npy"
IMAGE_TEST_PATH = OUTPUT_DIR / "test_image_embeddings.npy"

LIGHTGBM_PARAMS = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.03,
    "num_leaves": 512,
    "feature_fraction": 0.7,
    "bagging_fraction": 0.7,
    "bagging_freq": 1,
    "min_data_in_leaf": 20,
    "lambda_l1": 0.05,
    "lambda_l2": 0.2,
    "max_bin": 255,
    "device_type": "gpu",
    "gpu_platform_id": 0,
    "gpu_device_id": 0,
    "num_threads": max(1, (os.cpu_count() or 1) - 1),
    "verbose": -1,
}

# Enforce GPU; fail fast if not available
try:
    _test_ds = lgb.Dataset(
        np.random.rand(64, 16).astype(np.float32),
        label=np.random.rand(64).astype(np.float32)
    )
    _ = lgb.train({**LIGHTGBM_PARAMS, "device_type": "gpu"}, _test_ds, num_boost_round=1)
    print("LightGBM GPU check: OK")
except Exception as e:
    raise RuntimeError(
        "LightGBM GPU is not available in this environment. Install a GPU-enabled LightGBM build "
        "(e.g., conda install -c conda-forge lightgbm) and ensure GPU drivers/OpenCL are installed."
    ) from e

for directory in (FEATURE_DIR, MODEL_DIR):
    directory.mkdir(exist_ok=True)

print("Dataset available:", DATA_DIR.exists())
print("Embeddings available:", TEXT_TRAIN_PATH.exists() and IMAGE_TRAIN_PATH.exists())

LightGBM GPU check: OK
Dataset available: True
Embeddings available: True


In [3]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (75000, 4)
Test shape: (75000, 3)


In [ ]:
train_text = np.load(TEXT_TRAIN_PATH)
test_text = np.load(TEXT_TEST_PATH)
train_img = np.load(IMAGE_TRAIN_PATH) if IMAGE_TRAIN_PATH.exists() else np.zeros((train_text.shape[0], 0))
test_img = np.load(IMAGE_TEST_PATH) if IMAGE_TEST_PATH.exists() else np.zeros((test_text.shape[0], 0))

import re
IPQ_REGEX = re.compile(r"(\d+\.?\d*)\s*(pack|ct|count|pcs|pc|piece|pieces)", re.IGNORECASE)

def extract_ipq(series: pd.Series) -> np.ndarray:
    values = []
    for text in series.fillna(""):
        match = IPQ_REGEX.search(text)
        values.append(float(match.group(1)) if match else 0.0)
    return np.array(values, dtype=np.float32)

train_ipq = extract_ipq(train_df["catalog_content"])
test_ipq = extract_ipq(test_df["catalog_content"])

ipq_scaler = StandardScaler()
train_ipq_scaled = ipq_scaler.fit_transform(train_ipq.reshape(-1, 1))
test_ipq_scaled = ipq_scaler.transform(test_ipq.reshape(-1, 1))

train_stats = np.stack([
    train_df["catalog_content"].fillna("").str.len().to_numpy(),
    train_df["catalog_content"].fillna("").str.count(r"\d").to_numpy(),
], axis=1)

test_stats = np.stack([
    test_df["catalog_content"].fillna("").str.len().to_numpy(),
    test_df["catalog_content"].fillna("").str.count(r"\d").to_numpy(),
], axis=1)

stats_scaler = StandardScaler()
train_stats_scaled = stats_scaler.fit_transform(train_stats)
test_stats_scaled = stats_scaler.transform(test_stats)

train_features = np.concatenate([train_text, train_img, train_stats_scaled, train_ipq_scaled], axis=1)
test_features = np.concatenate([test_text, test_img, test_stats_scaled, test_ipq_scaled], axis=1)

# Cast to float32 for better GPU throughput and smaller footprint
train_features = train_features.astype(np.float32, copy=False)
test_features = test_features.astype(np.float32, copy=False)

np.save(FEATURE_DIR / "train_features.npy", train_features)
np.save(FEATURE_DIR / "test_features.npy", test_features)
np.save(FEATURE_DIR / "train_price.npy", train_df["price"].to_numpy(dtype=np.float32))
print("Feature shapes:", train_features.shape, test_features.shape)

Feature shapes: (75000, 2051) (75000, 2051)


In [4]:
features = train_features
target = np.load(FEATURE_DIR / "train_price.npy")

# Ensure float32 going into GPU
features = features.astype(np.float32, copy=False)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros_like(target, dtype=np.float32)
models = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(features, target), start=1):
    train_data = lgb.Dataset(features[train_idx], label=target[train_idx])
    valid_data = lgb.Dataset(features[valid_idx], label=target[valid_idx])
    model = lgb.train(
        LIGHTGBM_PARAMS,
        train_data,
        num_boost_round=5000,
        valid_sets=[valid_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=200, verbose=True),
            lgb.log_evaluation(period=100),
        ],
    )
    preds = model.predict(features[valid_idx], num_iteration=model.best_iteration)
    oof_preds[valid_idx] = preds
    models.append(model)
    joblib.dump(model, MODEL_DIR / f"lightgbm_fold{fold}.joblib")

np.save(MODEL_DIR / "oof_predictions.npy", oof_preds)
print("Stored OOF predictions and fold models.")

NameError: name 'train_features' is not defined

In [ ]:
def smape(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 1e-9) -> float:
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    denom = np.where(denom < eps, eps, denom)
    return float(np.mean(np.abs(y_true - y_pred) / denom) * 100)

score = smape(target, oof_preds)
print(f"OOF SMAPE: {score:.2f}%")

In [ ]:
mean_best_iteration = int(np.mean([model.best_iteration for model in models]))
mean_best_iteration = max(mean_best_iteration, 100)
print(f"Using {mean_best_iteration} boosting rounds for final model")
final_data = lgb.Dataset(features, label=target)
final_model = lgb.train(LIGHTGBM_PARAMS, final_data, num_boost_round=mean_best_iteration)
joblib.dump(final_model, MODEL_DIR / "lightgbm_final.joblib")

submission_preds = final_model.predict(test_features)
submission_preds = np.clip(submission_preds, a_min=0.1, a_max=None)
submission = pd.DataFrame({"sample_id": test_df["sample_id"], "price": submission_preds})
submission.to_csv(SUBMISSION_PATH, index=False)
print("Submission saved to", SUBMISSION_PATH)